In [15]:
import pandas as pd

admission = pd.read_csv('data/raw/admission.csv')
patient = pd.read_csv('data/raw/patient.csv')
department = pd.read_csv('data/raw/department.csv')
ward = pd.read_csv( 'data/raw/ward.csv')
bed = pd.read_csv('data/raw/bed.csv')
disease = pd.read_csv('data/raw/disease.csv')
billing = pd.read_csv('data/raw/billing.csv')
billing_detail = pd.read_csv( 'data/raw/billing_detail.csv')
doctor = pd.read_csv( 'data/raw/doctor.csv')
diagnostic_test = pd.read_csv( 'data/raw/diagnostic_test.csv')
patient_diagnostic = pd.read_csv('data/raw/patient_diagnostic.csv')
insurance_provider = pd.read_csv ('data/raw/insurance_provider.csv')
patient_insurance = pd.read_csv('data/raw/patient_insurance.csv')

assert admission['patient_id'].isin(patient['patient_id']).all(), "orphan patient_id in admission"
assert admission['department_id'].isin(department['department_id']).all(), "orphan department_id"
assert admission['ward_id'].isin(ward['ward_id']).all(), "orphan ward_id"
assert admission['bed_id'].isin(bed['bed_id']).all(), "orphan bed_id"
assert billing['admission_id'].isin(admission['admission_id']).all(), "orphan admission_id in billing"

print("All foreign keys valid ")
print(f"Admissions: {len(admission)} | Patients: {len(patient)} | Departments: {len(department)}")

All foreign keys valid 
Admissions: 45000 | Patients: 30000 | Departments: 11


In [16]:
admission['admission_date'] = pd.to_datetime(admission['admission_date'])
admission['discharge_date'] = pd.to_datetime(admission['discharge_date'])
patient['date_of_birth'] = pd.to_datetime(patient['date_of_birth'])
billing['bill_date'] = pd.to_datetime(billing['bill_date'])

In [17]:
admission['length_of_stay'] = (admission['discharge_date'] - admission['admission_date']).dt.days

In [18]:
for col in ['admission_type', 'admission_status']:
    admission[col] = admission[col].str.strip().str.title()

print(admission['admission_type'].unique())
print(admission['admission_status'].unique())

['Emergency' 'Elective']
['Discharged']


In [19]:
billing_detail['reference_id'] = billing_detail['reference_id'].fillna(-1)  

In [21]:
core = (admission
    .merge(patient, on='patient_id', how='left')
    .merge(department, on='department_id', how='left')
    .merge(ward, on='ward_id', how='left', suffixes=('', '_ward'))
    .merge(bed, on='bed_id', how='left')
    .merge(disease, on='disease_id', how='left')
    .merge(billing, on='admission_id', how='left')
)

core.to_csv('data/processed/hospital_admissions_core.csv', index=False)
print(f"Final core table: {core.shape}")

Final core table: (45000, 36)


In [23]:
print(f"Missing values: {core.isnull().sum().sum() / (core.shape[0]*core.shape[1]) * 100:.2f}%")
print(f"Duplicate admissions: {core['admission_id'].duplicated().sum()}")
print(f"Date range: {core['admission_date'].min()} to {core['admission_date'].max()}")
print(f"Departments: {core['department_name'].nunique()} | Wards: {core['ward_name'].nunique()}")

Missing values: 0.00%
Duplicate admissions: 0
Date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00
Departments: 6 | Wards: 27


In [24]:
core.head()


,admission_id,admission_date,discharge_date,admission_type,admission_status,patient_id,department_id,ward_id_x,bed_id,disease_id,...,ward_id_y,disease_name,disease_category,bill_id,bill_date,total_amount,insurance_covered_amount,patient_payable_amount,payment_status,payment_mode
0,1,2020-02-25,2020-02-27,Emergency,Discharged,166,2,6,76,10,...,6,Anemia,Hematological,1,2025-05-26,68483,61634.7,6848.3,Paid,Insurance
1,2,2022-02-22,2022-03-04,Elective,Discharged,8622,5,21,302,11,...,21,Fracture Femur,Orthopedic,2,2023-11-19,70917,35458.5,35458.5,Paid,Insurance
2,3,2021-02-03,2021-02-09,Elective,Discharged,23976,1,2,11,9,...,2,Chronic Obstructive Pulmonary Disease,Respiratory,3,2023-02-20,28137,25323.3,2813.7,Pending,Insurance
3,4,2021-12-31,2022-01-05,Elective,Discharged,16635,2,10,128,1,...,10,Acute Myocardial Infarction,Cardiac,4,2021-09-01,80665,64532.0,16133.0,Pending,Insurance
4,5,2022-07-02,2022-07-07,Elective,Discharged,10654,3,11,157,7,...,11,Hypertension,Cardiac,5,2021-07-13,54920,27460.0,27460.0,Pending,Insurance


In [26]:
core.columns.tolist()

['admission_id',
 'admission_date',
 'discharge_date',
 'admission_type',
 'admission_status',
 'patient_id',
 'department_id',
 'ward_id_x',
 'bed_id',
 'disease_id',
 'length_of_stay',
 'gender',
 'date_of_birth',
 'blood_group',
 'city',
 'contact_number',
 'department_name',
 'department_type',
 'floor_number',
 'status',
 'ward_name',
 'ward_type',
 'total_beds',
 'department_id_ward',
 'bed_number',
 'bed_status',
 'ward_id_y',
 'disease_name',
 'disease_category',
 'bill_id',
 'bill_date',
 'total_amount',
 'insurance_covered_amount',
 'patient_payable_amount',
 'payment_status',
 'payment_mode']